In [1]:
import xarray as xr
import geopandas as gpd
import rioxarray
import glob

ERA5-Land Variables Description - 24 features

🌧️ Atmospheric Forcing (Water & Energy Inputs)

- tp – Total precipitation (m). Total rainfall and snowfall.

- ssrd – Surface solar radiation downward (J/m²). Incoming shortwave radiation at the surface.

- strd – Surface thermal radiation downward (J/m²). Incoming longwave radiation at the surface.

- d2m – 2-meter dewpoint temperature (K). Represents near-surface atmospheric moisture; useful for humidity and vapor pressure deficit calculations.

- t2m – 2-meter air temperature (K). Standard near-surface air temperature; key driver of evapotranspiration and drought development.

☀️ Surface Energy Balance

- ssr – Surface net solar (shortwave) radiation (J/m²). Net shortwave energy at the surface.

- str – Surface net thermal (longwave) radiation (J/m²). Net longwave radiation at the surface.

- skt – Skin temperature (K). Temperature of the land surface “skin”; responds quickly to radiation and soil moisture changes.

- slhf – Surface latent heat flux (J/m²). Energy used for evaporation and transpiration.

💧 Evaporation & Evaporative Demand

- e – Total evaporation (m). Sum of evaporation components (soil + vegetation + interception).

- evabs – Evaporation from bare soil (m). Water evaporated directly from soil surface.

- evavt – Evaporation from vegetation transpiration (m). Plant transpiration component of evapotranspiration.

- pev – Potential evaporation (m). Maximum possible evaporation under unlimited water supply.

🌱 Soil Moisture State (Water Storage)

- swvl1 – Volumetric soil water content layer 1 (0–7 cm, m³/m³). Surface soil moisture.

- swvl2 – Volumetric soil water content layer 2 (7–28 cm, m³/m³). Shallow root-zone soil moisture.

- swvl3 – Volumetric soil water content layer 3 (28–100 cm, m³/m³). Deeper root-zone soil moisture.

- swvl4 – Volumetric soil water content layer 4 (100–289 cm, m³/m³). Deep soil moisture storage.

🌡️ Soil Thermal State

- stl1 – Soil temperature layer 1 (0–7 cm, K). Temperature of the top soil layer.

- stl2 – Soil temperature layer 2 (7–28 cm, K). Temperature of the shallow subsurface layer.

❄️ Snow Processes

- sd – Snow depth (m of water equivalent). Total depth of snowpack.

- smlt – Snowmelt (m water equivalent). Amount of melted snow contributing to surface water.

🌊 Runoff & Drainage (Water Outputs)

- ro – Total runoff (m). Combined surface and subsurface runoff.

- ssro – Surface runoff (m). Direct runoff from precipitation/snowmelt at surface.

- sro – Subsurface runoff (m). Drainage and lateral flow below the surface.

Open all monthly files as ONE dataset

In [ ]:
# ds = xr.open_dataset("D:\\Documents\\thesis\\data\\era5-land-all-variables\\era5_2025_01_week1.nc")

In [2]:
import xarray as xr
import glob

files = glob.glob(r"D:\\Documents\\thesis\\data\\era5-land-all-variables3\\*.nc")

bad = []
for f in files:
    try:
        xr.open_dataset(f).close()
    except Exception:
        bad.append(f)

print("Broken files:")
for b in bad:
    print(b)


Broken files:


In [3]:
files = glob.glob(r"D:\\Documents\\thesis\\data\\era5-land-all-variables3\\*.nc")
ds = xr.open_mfdataset(
    files,
    combine="by_coords",
    parallel=True,
    engine="h5netcdf"
)

Attach spatial reference (CRS)

In [4]:
ds = ds.rio.write_crs("EPSG:4326")

Load and prepare the Ebro basin shapefile

In [5]:
# Load the Ebro basin polygon from GeoPackage
ebro_shp = gpd.read_file(r"D:\\Documents\\thesis\\data\\ebro shapefile\\49642\\Results\\euhydro_ebro_v013.gpkg")

c:\Users\albat\anaconda3\envs\era5\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(
c:\Users\albat\anaconda3\envs\era5\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(
c:\Users\albat\anaconda3\envs\era5\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(
c:\Users\albat\anaconda3\envs\era5\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'euhydro_ebro_v013.gpkg': 'Canals_l' (default), 'Canals_p', 'Coastal_p', 'Culverts', 'Ditches_l', 'Ditches_p', 'InlandWater', 'Nodes', 'River_Net_l', 'River_Net_p', 'RiverBasins', 'Transit_p'. Specify layer pa

In [6]:
# Check CRS
print(ebro_shp.crs)
ebro_shp = ebro_shp.to_crs("EPSG:4326")

EPSG:3035


Clip ERA5-Land to the Ebro basin

In [7]:
ds = ds.rio.write_crs("EPSG:4326")

ds_ebro = ds.rio.clip(
    ebro_shp.geometry,
    ebro_shp.crs,
    drop=True
)

In [8]:
ds_ebro.data_vars

Data variables:
    d2m      (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    t2m      (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    skt      (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    stl1     (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    stl2     (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    sd       (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    smlt     (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    swvl1    (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    swvl2    (valid_time, latitude, longitude) float32 524MB dask.array<

Unit conversions

Temperature K → °C

In [9]:
ds_ebro["t2m"] = ds_ebro["t2m"] - 273.15
ds_ebro["d2m"] = ds_ebro["d2m"] - 273.15
ds_ebro["skt"] = ds_ebro["skt"] - 273.15
ds_ebro["stl1"] = ds_ebro["stl1"] - 273.15
ds_ebro["stl2"] = ds_ebro["stl2"] - 273.15

Water fluxes: m → mm

In [10]:
water_flux_vars = ["tp","pev","e","evabs","evavt","ro","sro","ssro","smlt"]
for v in water_flux_vars:
    ds_ebro[v] = ds_ebro[v] * 1000

# Evaporation variables are negative upward fluxes -> take absolute value
for v in ["e","evabs","evavt","pev"]:
    ds_ebro[v] = -ds_ebro[v]

Energy flux: J m⁻² → W m⁻²

In [11]:
energy_flux = ["slhf","ssr","ssrd","str","strd"]
for v in energy_flux:
    ds_ebro[v] = ds_ebro[v] / 3600

Aggregate hourly → daily

In [12]:
ds_ebro

<xarray.Dataset> Size: 13GB
Dimensions:      (valid_time: 314832, latitude: 16, longitude: 26)
Coordinates:
  * valid_time   (valid_time) datetime64[ns] 3MB 1990-01-01 ... 2025-11-30T23...
  * latitude     (latitude) float64 128B 42.2 42.1 42.0 41.9 ... 40.9 40.8 40.7
  * longitude    (longitude) float64 208B -1.6 -1.5 -1.4 -1.3 ... 0.7 0.8 0.9
    number       int64 8B 0
    expver       (valid_time) <U4 5MB dask.array<chunksize=(168,), meta=np.ndarray>
    spatial_ref  int64 8B 0
Data variables: (12/24)
    d2m          (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    t2m          (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    skt          (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    stl1         (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    stl2         (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    sd           (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    ...           ...
    pev          (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    ro           (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    ssro         (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    sro          (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    e            (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
    tp           (valid_time, latitude, longitude) float32 524MB dask.array<chunksize=(168, 16, 26), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-10T00:27 GRIB to CDM+CF via cfgrib-0.9.1...

In [13]:
ds_ebro = ds_ebro.rename({"valid_time": "time"})
sum_vars = ["tp","pev","e","evabs","evavt","ro","sro","ssro","smlt",
            "slhf","ssr","ssrd","str","strd"]

mean_vars = ["t2m","d2m","skt","stl1","stl2","sd",
             "swvl1","swvl2","swvl3","swvl4"]
ds_daily = xr.Dataset()

for v in sum_vars:
    ds_daily[v] = ds_ebro[v].resample(time="1D").sum()

for v in mean_vars:
    ds_daily[v] = ds_ebro[v].resample(time="1D").mean()

In [15]:
ds_daily = ds_daily.compute()

Basin-average time series

In [17]:
ds_daily_mean = ds_daily.mean(dim=["latitude", "longitude"])

Save files

In [19]:
ds_daily.to_netcdf(r"D:\\Documents\\thesis\\processed_data\\era5-land-all-variables\\era5land_ebro_daily3.nc")

ds_daily_mean.to_netcdf(r"D:\\Documents\\thesis\\processed_data\\era5-land-all-variables\\era5land_ebro_daily_mean3.nc")